# Loan Application (Diaz et al.) — RL Analysis

Análisis del agente PPO sobre el grafo DCR de Loan Application (Diaz et al. DEC2H 2024).

**Objetivo**: comprobar si 50k steps es suficiente para convergencia y comparar el frente Pareto con los resultados exactos de Diaz et al. (CSP+COP).

**Resultados de referencia (Diaz et al., slide 27):**

| Traza | Coste | Duración |
|---|---|---|
| provide doc → check → reject | 170.1 | 400 |
| provide doc → check → external rating → grant | 470.1 | 520 |
| provide doc → check → extra info → provide doc → check → grant | 340.2 | 1180 |

In [1]:
import re, csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── CONFIG ──────────────────────────────────────────────────────────────────
LOGS_DIR  = Path("/Users/sofia/Desktop/loanapp_logs")
SMOOTH_W  = 50   # smaller window — fewer episodes with 50k steps
# ────────────────────────────────────────────────────────────────────────────

COLORS = {
    "α=0.0 β=0.0": "#2196F3",
    "α=1.0 β=0.0": "#4CAF50",
    "α=0.0 β=1.0": "#F44336",
    "α=0.5 β=0.5": "#FF9800",
    "α=2.0 β=0.5": "#9C27B0",
    "α=0.5 β=2.0": "#00BCD4",
}

# Diaz et al. reference Pareto points (CSP+COP exact method)
DIAZ_PARETO = [
    {"label": "Diaz (reject path)",          "cost": 170.1, "duration": 400},
    {"label": "Diaz (external rating path)", "cost": 470.1, "duration": 520},
    {"label": "Diaz (extra info loop)",      "cost": 340.2, "duration": 1180},
]

def parse_weights(exp_id):
    m = re.search(r'_a([\dp]+)_b([\dp]+)', exp_id)
    if m:
        return float(m.group(1).replace("p",".")), float(m.group(2).replace("p","."))
    return None, None

def wlabel(a, b):
    return f"α={a:.1f} β={b:.1f}" if a is not None else "unknown"

def smooth(v, w):
    if w <= 1 or len(v) < w: return np.array(v)
    return np.convolve(v, np.ones(w)/w, mode='valid')

def load_data(logs_dir):
    rows = []
    for p in sorted(logs_dir.glob("train_trace_exp_LoanApp_Diaz_*.csv")):
        exp_id = p.stem.replace("train_trace_exp_","")
        a, b = parse_weights(exp_id)
        label = wlabel(a, b)
        with open(p, newline="") as f:
            for row in csv.DictReader(f):
                if str(row.get("done","")).lower() not in ("true","1"):
                    continue
                try:
                    rows.append({
                        "label":     label, "alpha": a, "beta": b,
                        "episode":   int(row.get("episode", 0)),
                        "timestep":  int(row.get("global_timestep", 0)),
                        "reward":    float(row.get("ep_rew_sum", 0)),
                        "steps":     int(row.get("episode_steps", 0)),
                        "cost":      float(row["episode_cost"])     if row.get("episode_cost")     not in ("","None",None) else None,
                        "duration":  float(row["episode_duration"]) if row.get("episode_duration") not in ("","None",None) else None,
                        "accepting": str(row.get("accepting","")).lower() in ("true","1"),
                    })
                except (ValueError, KeyError):
                    pass
    return pd.DataFrame(rows)

df = load_data(LOGS_DIR)
if df.empty:
    print(f"⚠️  No hay CSVs de LoanApp_Diaz en {LOGS_DIR}")
    print("   Ejecuta primero el scp del cluster.")
else:
    n_pairs = df["label"].nunique()
    print(f"✓ {len(df):,} episodios · {n_pairs} weight pairs cargados")
    print(f"  Timesteps max: {df['timestep'].max():,}")
    display(df.groupby("label")[["episode","timestep"]].max())

✓ 34,923 episodios · 6 weight pairs cargados
  Timesteps max: 51,200


,episode,timestep
label,,
α=0.0 β=0.0,11760,51198
α=0.0 β=1.0,3290,51198
α=0.5 β=0.5,7518,51197
α=0.5 β=2.0,814,51027
α=1.0 β=0.0,10625,51200
α=2.0 β=0.5,916,51037


## 1 · Convergencia de reward — ¿son suficientes 50k steps?

Si las curvas se estabilizan antes del 100%, el agente converge con 50k steps.
El eje X está normalizado a 0–100% para comparar velocidad entre weight pairs.

In [2]:
fig = go.Figure()
for label, grp in df.groupby("label"):
    vals = grp["reward"].values
    color = COLORS.get(label, "#888")
    n = len(vals)
    x_pct = np.linspace(0, 100, n)
    fig.add_trace(go.Scatter(x=x_pct, y=vals, mode="lines",
        line=dict(color=color, width=1), opacity=0.15,
        showlegend=False, hoverinfo="skip"))
    sv = smooth(vals, SMOOTH_W)
    offset = n - len(sv)
    sx_pct = np.linspace(offset / n * 100, 100, len(sv))
    fig.add_trace(go.Scatter(x=sx_pct, y=sv, mode="lines",
        name=label, line=dict(color=color, width=2.5)))

fig.update_layout(
    title="Reward convergence — LoanApp Diaz (50k steps)",
    xaxis_title="Training progress (%)", yaxis_title="Episode reward",
    xaxis=dict(ticksuffix="%", range=[0,100], showgrid=True, gridcolor="#eee"),
    yaxis=dict(showgrid=True, gridcolor="#eee"),
    plot_bgcolor="white", height=460, legend_title="α=cost, β=duration")
fig.show()

## 2 · Pareto front — RL vs Diaz et al. (CSP+COP)

Los puntos negros son los resultados exactos de Diaz et al. obtenidos con CSP+COP.
Las estrellas de colores son los puntos Pareto encontrados por el agente RL.

In [5]:
def dominates(a, b):
    return a[0] <= b[0] and a[1] <= b[1] and (a[0] < b[0] or a[1] < b[1])

def pareto_front(points):
    seen, unique = {}, []
    for p in points:
        k = (round(p["cost"],2), round(p["duration"],2))
        if k not in seen:
            seen[k] = p; unique.append(p)
    front = [c for c in unique if not any(
        dominates((p["cost"],p["duration"]),(c["cost"],c["duration"]))
        for p in unique if p is not c)]
    return sorted(front, key=lambda x: x["cost"])

acc = df[df["accepting"] & df["cost"].notna() & df["duration"].notna()]
acc_no_baseline = acc[~((acc["alpha"]==0.0) & (acc["beta"]==0.0))]
points = acc_no_baseline[["label","cost","duration","steps","episode","timestep"]].to_dict("records")
front  = pareto_front(points)

fig = go.Figure()

# Background scatter — all accepting traces
for label, grp in acc.groupby("label"):
    fig.add_trace(go.Scatter(
        x=grp["cost"], y=grp["duration"], mode="markers",
        marker=dict(color=COLORS.get(label,"#888"), size=4, opacity=0.2),
        name=label, showlegend=True,
        hovertemplate=f"<b>{label}</b><br>cost=%{{x:.1f}}<br>dur=%{{y:.1f}}<extra></extra>"))

# RL Pareto front
if front:
    fx = [p["cost"] for p in front]
    fy = [p["duration"] for p in front]
    fh = [f"<b>RL Pareto</b><br>cost={p['cost']:.1f}<br>dur={p['duration']:.1f}<br>{p['label']}" for p in front]
    fig.add_trace(go.Scatter(x=fx, y=fy, mode="markers+lines",
        name="RL Pareto front",
        marker=dict(color="black", size=12, symbol="star"),
        line=dict(color="black", width=2, dash="dot"),
        hovertemplate="%{customdata}<extra></extra>", customdata=fh))

# Diaz et al. reference points
dx = [p["cost"] for p in DIAZ_PARETO]
dy = [p["duration"] for p in DIAZ_PARETO]
dh = [f"<b>{p['label']}</b><br>cost={p['cost']}<br>dur={p['duration']}" for p in DIAZ_PARETO]
fig.add_trace(go.Scatter(x=dx, y=dy, mode="markers",
    name="Diaz et al. (CSP+COP)",
    marker=dict(color="red", size=14, symbol="diamond", line=dict(color="darkred", width=2)),
    hovertemplate="%{customdata}<extra></extra>", customdata=dh))

fig.update_layout(
    title="Pareto Front — RL (PPO) vs Diaz et al. (CSP+COP) · LoanApp",
    xaxis_title="Total Cost", yaxis_title="Total Duration",
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#eee"),
    yaxis=dict(showgrid=True, gridcolor="#eee"),
    height=520, legend_title="Method / Weight pair")
fig.show()

print(f"\nRL Pareto front: {len(front)} puntos")
print(f"Diaz et al.:     {len(DIAZ_PARETO)} puntos (exactos)\n")
pd.DataFrame(front)[["label","cost","duration","steps"]].sort_values("cost")


RL Pareto front: 1 puntos
Diaz et al.:     3 puntos (exactos)



,label,cost,duration,steps
0,α=0.0 β=1.0,170.1,400.0,4


## 3 · Secuencias de las trazas Pareto-óptimas

Muestra paso a paso qué eventos ejecutó el agente en cada traza Pareto.

In [4]:
def load_episode_sequence(logs_dir, label, episode_num):
    for p in sorted(logs_dir.glob("train_trace_exp_LoanApp_Diaz_*.csv")):
        exp_id = p.stem.replace("train_trace_exp_","")
        a, b = parse_weights(exp_id)
        if wlabel(a,b) != label:
            continue
        steps = []
        with open(p, newline="") as f:
            for row in csv.DictReader(f):
                if int(row.get("episode",0)) == episode_num:
                    steps.append({
                        "step":     int(row.get("step_in_episode", len(steps)+1)),
                        "label":    row.get("action_label",""),
                        "reward":   float(row.get("reward",0)),
                        "cost":     row.get("event_cost",""),
                        "duration": row.get("event_duration",""),
                        "accepting":str(row.get("accepting","")).lower() in ("true","1"),
                    })
                elif steps:
                    break
        if steps:
            return steps
    return []

print("Trazas Pareto-óptimas del agente RL:\n")
for p in front:
    seq = load_episode_sequence(LOGS_DIR, p["label"], p["episode"])
    print(f"{'─'*65}")
    print(f"Weight: {p['label']}  |  cost={p['cost']:.1f}  dur={p['duration']:.1f}  steps={len(seq)}")
    print(f"{'─'*65}")
    print(f"  {'#':>2}  {'Event':<35} {'Cost':>6} {'Dur':>6}  Reward")
    for s in seq:
        marker = "✓" if s["accepting"] else " "
        c = f"{float(s['cost']):.1f}"     if s['cost']     not in ('','None',None) else '—'
        d = f"{float(s['duration']):.1f}" if s['duration'] not in ('','None',None) else '—'
        print(f"  {s['step']:>2}. [{marker}] {s['label']:<35} {c:>6} {d:>6}  {s['reward']:+.2f}")
    print()

# Comparación directa con Diaz
print("─"*65)
print("Referencia Diaz et al. (CSP+COP exacto):")
for p in DIAZ_PARETO:
    print(f"  cost={p['cost']:.1f}  dur={p['duration']:.0f}  → {p['label']}")

Trazas Pareto-óptimas del agente RL:

─────────────────────────────────────────────────────────────────
Weight: α=0.0 β=1.0  |  cost=170.1  dur=400.0  steps=4
─────────────────────────────────────────────────────────────────
   #  Event                                 Cost    Dur  Reward
   1. [ ] reject loan                           50.0  100.0  -10.00
   2. [ ] provide documentation                  0.1   60.0  -59.10
   3. [ ] check conditions                     120.0  240.0  -237.10
   4. [✓] reject loan                           50.0  100.0  +100.00

─────────────────────────────────────────────────────────────────
Referencia Diaz et al. (CSP+COP exacto):
  cost=170.1  dur=400  → Diaz (reject path)
  cost=470.1  dur=520  → Diaz (external rating path)
  cost=340.2  dur=1180  → Diaz (extra info loop)
